In [84]:
pip install openai

In [85]:
from openai import OpenAI
import os 
client = OpenAI(
    api_key = os.getenv(),
)

In [86]:
from google.colab import drive
drive.mount('/content/drive')



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [87]:
training_file_name = '/content/drive/MyDrive/fine_tune openai/training_data.jsonl'
validation_file_name = '/content/drive/MyDrive/fine_tune openai/validation_data.jsonl'

In [88]:
training_file_id = client.files.create(
  file = open(training_file_name, "rb"),
  purpose = "fine-tune"
)

validation_file_id = client.files.create(
  file = open(validation_file_name, "rb"),
  purpose = "fine-tune"
)

print(f"Training File ID: {training_file_id}")
print(f"Validation File ID: {validation_file_id}")

Training File ID: FileObject(id='file-ptJxQkcIFn286o6oJnTRXG6d', bytes=5598, created_at=1729539828, filename='training_data.jsonl', object='file', purpose='fine-tune', status='processed', status_details=None)
Validation File ID: FileObject(id='file-eSdfsX0PXvOsWWMfArUeddFC', bytes=2792, created_at=1729539829, filename='validation_data.jsonl', object='file', purpose='fine-tune', status='processed', status_details=None)


In [89]:
response = client.fine_tuning.jobs.create(
  training_file=training_file_id.id,
  validation_file=validation_file_id.id,
  model="gpt-3.5-turbo",
  hyperparameters={
    "n_epochs": 15,
	  "batch_size": 3,
	  "learning_rate_multiplier": 0.3
  }
)
job_id = response.id
status = response.status

print(f'Fine-tunning model with jobID: {job_id}.')
print(f"Training Response: {response}")
print(f"Training Status: {status}")

Fine-tunning model with jobID: ftjob-250S94XtZVHlhZh6RxdMEleK.
Training Response: FineTuningJob(id='ftjob-250S94XtZVHlhZh6RxdMEleK', created_at=1729539831, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(n_epochs=15, batch_size=3, learning_rate_multiplier=0.3), model='gpt-3.5-turbo-0125', object='fine_tuning.job', organization_id='org-E4AxAsw8JsiRQLswmIe7zBQK', result_files=[], seed=1215017330, status='validating_files', trained_tokens=None, training_file='file-ptJxQkcIFn286o6oJnTRXG6d', validation_file='file-eSdfsX0PXvOsWWMfArUeddFC', estimated_finish=None, integrations=[], user_provided_suffix=None)
Training Status: validating_files


In [90]:
import signal
import datetime


def signal_handler(sig, frame):
    status = client.fine_tuning.jobs.retrieve(job_id).status
    print(f"Stream interrupted. Job is still {status}.")
    return

print(f"Streaming events for the fine-tuning job: {job_id}")

signal.signal(signal.SIGINT, signal_handler)

events = client.fine_tuning.jobs.list_events(fine_tuning_job_id=job_id)
try:
    for event in events:
        print(
            f'{datetime.datetime.fromtimestamp(event.created_at)} {event.message}'
        )
except Exception:
    print("Stream interrupted (client disconnected).")

Streaming events for the fine-tuning job: ftjob-250S94XtZVHlhZh6RxdMEleK
2024-10-21 19:43:51 Validating training file: file-ptJxQkcIFn286o6oJnTRXG6d and validation file: file-eSdfsX0PXvOsWWMfArUeddFC
2024-10-21 19:43:51 Created fine-tuning job: ftjob-250S94XtZVHlhZh6RxdMEleK


In [91]:
import time

status = client.fine_tuning.jobs.retrieve(job_id).status
if status not in ["succeeded", "failed"]:
    print(f"Job not in terminal status: {status}. Waiting.")
    while status not in ["succeeded", "failed"]:
        time.sleep(2)
        status = client.fine_tuning.jobs.retrieve(job_id).status
        print(f"Status: {status}")
else:
    print(f"Finetune job {job_id} finished with status: {status}")

Job not in terminal status: validating_files. Waiting.
Status: validating_files
Status: validating_files
Status: validating_files
Status: validating_files
Status: validating_files
Status: queued
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Statu

In [92]:
result = client.fine_tuning.jobs.list()
print(f"Found {len(result.data)} finetune jobs.")

Found 7 finetune jobs.


In [93]:
fine_tuned_model = result.data[0].fine_tuned_model

In [94]:
print(fine_tuned_model)

ft:gpt-3.5-turbo-0125:personal::AKsiAYM6


In [101]:
answer = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[
        {"role": "user", "content": "ما هي أنواع الكتب التي تقدمها؟"},
    ]
)

print(answer.choices[0].message)

ChatCompletionMessage(content='نوع الكتب التي أقدمها يعتمد على ما ترغب في معرفته أو الاطلاع عليه. يمكنني تزويدك بمعلومات حول العديد من أنواع الكتب مثل الأدبية، العلمية، التاريخية، الدينية، الفلسفية، السير والسير الذاتية، كتب التنمية البشرية، كتب الطهي والطعام، والعديد من الأنواع الأخرى. كما يمكنني توجيهك إلى كتب متخصصة في مجالات معينة مثل الصحة، التكنولوجيا، العلاقات الإنسانية، التسويق، وغيرها. يرجى تحديد نوع الكتب التي تفضلها حتى أتمكن من توجيهك بشكل أفضل.', refusal=None, role='assistant', audio=None, function_call=None, tool_calls=None)


In [102]:
answer = client.chat.completions.create(
    model=fine_tuned_model,
    messages=[
        {"role": "user", "content": "ما هي أنواع الكتب التي تقدمها؟"},
    ]
)

print(answer.choices[0].message)

ChatCompletionMessage(content='نوع الكتب التي أقوم بتقديمها تتنوع بشكل كبير وتشمل مجموعة متنوعة من الأجزاء من الخيال والرومانسية والخيال العلمي والرعب والدراما والتاريخ والسيرة الذاتية والأعمال والاقتصاد وتطوير الذات وغيرها الكثير. من المؤكد أن هناك نوعًا من الكتب تناسب كل زوق واهتمام.', refusal=None, role='assistant', audio=None, function_call=None, tool_calls=None)


In [103]:
answer = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[
        {"role": "user", "content": "هل يمكنك أن توصي بكتاب جيد للأطفال من سن 8 إلى 10 سنوات"},
    ]
)

print(answer.choices[0].message)

ChatCompletionMessage(content='نعم ، إليك بعض الكتب الموصى بها للأطفال من سن 8 إلى 10 سنوات:\n\n1. "هاري بوتر وحجر الفيلسوف" للكاتبة ج. ك. رولينج - سلسلة كتب هاري بوتر تعتبر من الكلاسيكيات للأطفال في هذا العمر.\n2. "العجوز الذي قفز من نافذة واختفى" للكاتب جوناس يونسون - قصة كوميدية مؤثرة تحكي قصة رجل مسن يقرر الهروب من دار المسنين.\n3. "زرافة في الثلاجة" للكاتب ديفيد والريم - قصة مضحكة وممتعة تدور حول زرافة تعيش في ثلاجة.\n4. "الأميرة والتنين" للكاتبة جوليا دونالدسون - قصة فانتازية تحكي قصة الأميرة التي تنقذ المملكة من التنين المخيف.\n\nهذه بعض الكتب التي يمكن أن تكون مناسبة للأطفال في هذا العمر. يمكنك البحث أيضاً عن كتب أخرى تناسب اهتمامات الطفل مثل الألغاز، الحكايات الخيالية، الحيوانات، وغيرها.', refusal=None, role='assistant', audio=None, function_call=None, tool_calls=None)


In [104]:
answer = client.chat.completions.create(
    model=fine_tuned_model,
    messages=[
        {"role": "user", "content": "هل يمكنك أن توصي بكتاب جيد للأطفال من سن 8 إلى 10 سنوات؟"},
    ]
)

print(answer.choices[0].message)

ChatCompletionMessage(content='بالتأكيد، إليك بعض الكتب التي يمكنني توصيتها للأطفال في هذا العمر:\n\n1. "هاري بوتر وحجر الفيلسوف" للكاتبة ج. ك. رولينج: سلسلة من الكتب المشهورة والمحبوبة على مستوى العالم، تلك الرواية الأولى تحكي قصة فتى يكتشف أنه ساحر.\n\n2. "شجرة البقدونس" للكاتبة إينيد بليتون: قصة تتحدث عن الإخوة جوليان وآن ذوي الخمسة أعوام، ومغامراتهم مع الدمى وتأجير أشياء من متجر شجرة البقدونس.\n\n3. "في بلاد القطط" للكاتبة يوهانا سباير: قصة فتى يذهب في رحلة تربية للقطط في كتاب مليء بالمغامرات والسحر.\n\n4. "سلسلة دايري أوف ويمبي كيدز" للكاتب تاي ريكارد: سلسلة من الكتب المثيرة التي تحكي قصة فتى لديه قدرات خاصة ويجب عليه حماية العالم من الشر.\n\n5. "سلسلة جريج" للكاتب جيف كيني: سلسلة كوميدية متنوعة تروي حكايات المراهق الطبيعي جريج هيفلي الذي يواجه الكثير من التحديات في المدرسة والمنزل.\n\nهؤلاء بعض الكتب التي يمكنني توصيتها للأطفال في هذا العمر، وهي غنية بالخيال والمغامرات والتعلم.', refusal=None, role='assistant', audio=None, function_call=None, tool_calls=None)


In [105]:
answer = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[
        {"role": "user", "content": "كيف يمكنني الاشتراك في النشرة الإخبارية لتلقي التحديثات؟"},
    ]
)

print(answer.choices[0].message)

ChatCompletionMessage(content='يمكنك الاشتراك في النشرة الإخبارية لتلقي التحديثات عن طريق زيارة موقع الشركة أو الجهة الإعلامية التي ترغب في الاشتراك في نشرتها الإخبارية. عادة ما يوجد خيار للاشتراك عبر إدخال عنوان بريدك الإلكتروني وتأكيد الاشتراك. كما يمكنك أيضا البحث عن خانة اشتراك في النشرة الإخبارية على موقع الويب الخاص بالشركة أو الجهة الإعلامية وتعبئة النموذج المطلوب.\n\nبعد الاشتراك، ستبدأ في تلقي التحديثات والأخبار الجديدة مباشرة على بريدك الإلكتروني المسجل في النشرة الإخبارية. يمكنك أيضا إلغاء الاشتراك في أي وقت إذا كنت لا ترغب بتلقي المزيد من الرسائل الإخبارية.', refusal=None, role='assistant', audio=None, function_call=None, tool_calls=None)


In [106]:
answer = client.chat.completions.create(
    model=fine_tuned_model,
    messages=[
        {"role": "user", "content": "كيف يمكنني الاشتراك في النشرة الإخبارية لتلقي التحديثات؟"},
    ]
)

print(answer.choices[0].message)

ChatCompletionMessage(content='يمكنك الاشتراك في النشرة الإخبارية عن طريق زيارة موقع الشركة أو المنظمة التي ترغب في الاشتراك في نشرتها الإخبارية وإدخال عنوان بريدك الإلكتروني في نموذج الاشتراك. قد تجد أيضًا خيارًا للاشتراك عند تسجيل الدخول إلى حسابك على الموقع.', refusal=None, role='assistant', audio=None, function_call=None, tool_calls=None)
